In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)
test_dataset = torchvision.datasets.MNIST(
    root = './data',
    train = False,
    transform = transform,
    download = True
)
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size = 64,shuffle = True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size = 64,shuffle = False)

In [ ]:
# CNN

class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv1 = nn.Conv2d(1,32,3)
    self.pool = nn.MaxPool2d(2,2)

    self.conv2 = nn.Conv2d(32,64,3)
    self.fc1 = nn.Linear(64*5*5,128)
    self.fc2 = nn.Linear(128,10)
  def forward(self,x):
    x = self.pool(torch.relu(self.conv1(x)))
    x = self.pool(torch.relu(self.conv2(x)))

    x = x.view(-1,64*5*5)

    x = torch.relu(self.fc1(x))
    x = self.fc2(x)
    return x


In [ ]:
  #RNN
  class RNN(nn.Module):
    def __init__(self):
      super(RNN,self).__init__()
      self.rnn = nn.RNN(input_size= 28,hidden_size= 128,num_layers = 2,batch_first = True)
      self.fc = nn.Linear(128,10)
    def forward(self,x):
      x = x.squeeze(1)
      x, _ = self.rnn(x)
      x = self.fc(x[:,-1,:])
      return x


In [ ]:
# Model Initialize
cnn_model = CNN()
rnn_model = RNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(),lr = 0.001)



In [ ]:
#TRAINING LOOP

epochs = 5

cnn_model.train()

for epoch in range(epochs):
  running_loss = 0

  for images,labels in train_loader:
    optimizer.zero_grad()

    outputs = cnn_model(images)

    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")


Epoch 1/5, Loss: 0.18664023486712675
Epoch 2/5, Loss: 0.054196159395554076
Epoch 3/5, Loss: 0.03710748571677874
Epoch 4/5, Loss: 0.02807747125432619
Epoch 5/5, Loss: 0.021450815083597898


In [ ]:
rnn_model = RNN()
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)  # ✅ new optimizer

for epoch in range(5):
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = rnn_model(images)   # ✅ RNN use
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

In [ ]:
epochs = 5
rnn_model.train()

for epoch in range(epochs):
  running_loss = 0

  for images,labels in train_loader:
    optimizer.zero_grad()

    outputs = rnn_model(images)

    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")

Epoch 1/5, Loss: 0.14148737914038143
Epoch 2/5, Loss: 0.1159381063268788
Epoch 3/5, Loss: 0.1249738827676399
Epoch 4/5, Loss: 0.1189680136938784
Epoch 5/5, Loss: 0.10847489757320361


In [ ]:
# ACCURACY Check
def calculate_accuracy(model,dataloader):
  correct = 0
  total = 0

  model.eval()
  with torch.no_grad():
    for images,labels in dataloader:
      outputs = model(images)

      _,predicted = torch.max(outputs.data,1)

      total += labels.size(0)

      correct += (predicted == labels).sum().item()

    print(f"Accuracy of the model on the test images: {100*correct/total}%")



In [ ]:
calculate_accuracy(cnn_model, test_loader)
calculate_accuracy(rnn_model, test_loader)

Accuracy of the model on the test images: 99.09%
Accuracy of the model on the test images: 97.13%
